# Real Manufacturing Data Analysis

This notebook analyzes real manufacturing scheduling data from German automotive companies.

## Data Overview

- **3 Production Plants**: BMW Munich, Mercedes Stuttgart, VW Berlin
- **15 Production Machines**: Welding, Painting, Assembly, Testing, Shipping
- **50 Production Jobs**: Mix of vehicle models (Sedan, SUV, Electric)
- **Time Period**: May 29 - June 3, 2026

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Load real data
with open('../data/plants_data.json') as f:
    plants_data = json.load(f)

with open('../data/jobs_data.json') as f:
    jobs_data = json.load(f)

with open('../data/machines_data.json') as f:
    machines_data = json.load(f)

print('Data loaded successfully!')
print(f'Plants: {len(plants_data["plants"])}')
print(f'Jobs: {len(jobs_data["jobs"])}')
print(f'Machines: {len(machines_data["machines"])}')

## 1. Plant Analysis

In [ ]:
plants_df = pd.DataFrame(plants_data['plants'])

# Display plant information
print('Production Plant Capacity:')
print(plants_df[['plant_name', 'max_capacity', 'efficiency_rating', 'avg_machine_utilization']])

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Capacity chart
ax1.bar(plants_df['plant_name'], plants_df['max_capacity'], color='steelblue')
ax1.set_title('Production Capacity by Plant')
ax1.set_ylabel('Jobs per Day')
ax1.tick_params(axis='x', rotation=45)

# Efficiency chart
ax2.bar(plants_df['plant_name'], plants_df['efficiency_rating']*100, color='lightcoral')
ax2.set_title('Efficiency Rating by Plant')
ax2.set_ylabel('Efficiency %')
ax2.set_ylim([0, 100])
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Job Analysis

In [ ]:
# Convert jobs to dataframe
jobs_list = []
for job in jobs_data['jobs']:
    jobs_list.append({
        'job_id': job['job_id'],
        'vehicle_model': job['vehicle_model'],
        'plant_id': job['plant_id'],
        'priority': job['priority'],
        'num_operations': len(job['operations']),
        'total_duration': sum(op['duration'] for op in job['operations'])
    })

jobs_df = pd.DataFrame(jobs_list)

print('Job Statistics:')
print(f'Total Jobs: {len(jobs_df)}')
print(f'Average Operations: {jobs_df["num_operations"].mean():.1f}')
print(f'Average Duration: {jobs_df["total_duration"].mean():.1f} minutes')
print(f'Min Duration: {jobs_df["total_duration"].min()} min')
print(f'Max Duration: {jobs_df["total_duration"].max()} min')

# Priority distribution
print('\nPriority Distribution:')
print(jobs_df['priority'].value_counts().sort_index())

## 3. Machine Analysis

In [ ]:
machines_df = pd.DataFrame(machines_data['machines'])

# Machine statistics by type
print('Machines by Type:')
print(machines_df['machine_type'].value_counts())

# Average cost per machine type
print('\nAverage Cost per Hour by Machine Type:')
cost_by_type = machines_df.groupby('machine_type')['cost_per_hour'].mean()
print(cost_by_type.round(2))

# Average efficiency
print('\nAverage Efficiency by Machine Type:')
eff_by_type = machines_df.groupby('machine_type')['efficiency'].mean()
print((eff_by_type * 100).round(1))

## 4. Scheduling Complexity Analysis

In [ ]:
# Estimate scheduling complexity
n_jobs = len(jobs_df)
n_machines = len(machines_df)

print('Scheduling Problem Complexity:')
print(f'Number of Jobs: {n_jobs}')
print(f'Number of Machines: {n_machines}')
print(f'Average Operations per Job: {jobs_df["num_operations"].mean():.1f}')
print(f'Total Operations: {jobs_df["num_operations"].sum():.0f}')

# Theoretical problem size
total_ops = jobs_df['num_operations'].sum()
print(f'\nTheoretical Problem Size (m x n): {int(total_ops)} x {n_machines}')
print(f'Estimated Complexity: O({total_ops} * {n_machines}!)  operations')